In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
#Funciones auxiliares sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold #Split y cross Validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score #Metricas
from sklearn.utils import shuffle 


#Gradient Boosting
import lightgbm as lgb


# Configuramos el estilo de las visualizaciones
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

breed_df = pd.read_csv('C:/Users/GWF/Documents/GitHub/UA_MDM_Labo2/input/petfinder-adoption-prediction/breed_labels.csv')
color_df = pd.read_csv('C:/Users/GWF/Documents/GitHub/UA_MDM_Labo2/input/petfinder-adoption-prediction/color_labels.csv')
state_df = pd.read_csv('C:/Users/GWF/Documents/GitHub/UA_MDM_Labo2/input/petfinder-adoption-prediction/state_labels.csv')

# Definimos la ruta
ruta_archivo = 'C:/Users/GWF/Documents/GitHub/UA_MDM_Labo2/input/petfinder-adoption-prediction/train/train.csv'

# Intentamos leer con latin-1

# Agregamos el parámetro sep=';'
train_df = pd.read_csv(ruta_archivo, sep=',', encoding='latin-1')


In [2]:
SEED = 42 #Semilla de procesos aleatorios (para poder replicar exactamente al volver a correr un modelo)
TEST_SIZE = 0.2 #Facción para train/test= split

In [3]:
#Separo un 20% para test estratificado opr target
df_train, df_test = train_test_split(train_df,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = train_df.AdoptionSpeed)

In [4]:
import pandas as pd

# 1. Cargar el dataset original
# Asegúrate de que el archivo esté en la misma carpeta que tu notebook

def categorizar_tamano(row):
    breed = str(row['BreedName']).lower()
    pet_type = row['Type']
    
    # Gatos (Type = 2): Los clasificamos como 'Pequeño' por defecto en comparación a un perro
    if pet_type == 2:
        return 'Pequeño'
        
    # Perros (Type = 1)
    if 'mixed breed' in breed:
        return 'Mediano' # Promedio estadístico seguro para perros mestizos
        
    # A. Excepciones específicas (se evalúan antes que las reglas generales)
    excepciones_grandes = ['black russian terrier', 'airedale terrier', 'afghan hound', 'bloodhound', 'greyhound', 'irish wolfhound']
    excepciones_medianos = ['american staffordshire terrier', 'pit bull terrier', 'bull terrier', 'beagle', 'basset hound', 'whippet', 'standard schnauzer', 'cocker spaniel']
    excepciones_pequenos = ['dachshund', 'miniature schnauzer', 'toy', 'miniature']
    
    if any(exc in breed for exc in excepciones_grandes):
        return 'Grande'
    if any(exc in breed for exc in excepciones_medianos):
        return 'Mediano'
    if any(exc in breed for exc in excepciones_pequenos):
        return 'Pequeño'
        
    # B. Reglas generales por familias funcionales / palabras clave
    # La mayoría de los terriers y perros de compañía son pequeños
    keywords_pequenos = ['terrier', 'chihuahua', 'pomeranian', 'pug', 'shih tzu', 'maltese', 'pekingese', 'corgi', 'bichon', 'french bulldog', 'papillon', 'havanese', 'pinscher']
    
    # La mayoría de los pastores, cobradores, mastines y sabuesos son grandes
    keywords_grandes = ['retriever', 'shepherd', 'mastiff', 'great', 'husky', 'malamute', 'akita', 'hound', 'pointer', 'setter', 'collie', 'rottweiler', 'doberman', 'boxer', 'american bulldog', 'sheepdog', 'dane', 'bernard', 'pyrenees', 'newfoundland', 'weimaraner', 'ridgeback', 'corso', 'komondor', 'kuvasz', 'borzoi', 'dalmatian']
    
    if any(kw in breed for kw in keywords_pequenos):
        return 'Pequeño'
    if any(kw in breed for kw in keywords_grandes):
        return 'Grande'
        
    # C. Todo lo que no caiga en las reglas anteriores se asume Mediano (ej. Spaniels, Bulldogs ingleses, Spitz)
    return 'Mediano'

# 2. Aplicar la función para crear la nueva columna
breed_df['SizeCategory'] = breed_df.apply(categorizar_tamano, axis=1)

# Unimos indicando que Breed1 equivale a BreedID
df_train = df_train.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

df_test = df_test.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

# Opcional: eliminar la columna BreedID duplicada que queda tras el merge
df_train = df_train.drop(columns=['BreedID'])
df_test = df_test.drop(columns=['BreedID'])
# 3. Exportar el resultado a un nuevo archivo CSV
nuevo_archivo = 'PetFinder-BreedLabels-Sized.csv'
breed_df.to_csv(nuevo_archivo, index=False)

print(f"Archivo generado exitosamente: '{nuevo_archivo}'")
print("\nMuestra de cómo quedaron las categorías:")
display(breed_df.sample(10, random_state=42))

Archivo generado exitosamente: 'PetFinder-BreedLabels-Sized.csv'

Muestra de cómo quedaron las categorías:


,BreedID,Type,BreedName,SizeCategory
183,184,1,Pumi,Mediano
60,61,1,Chinese Crested Dog,Mediano
124,125,1,Irish Wolfhound,Grande
93,94,1,Fila Brasileiro,Mediano
63,64,1,Chocolate Labrador Retriever,Grande
9,10,1,American Staffordshire Terrier,Mediano
147,148,1,Manchester Terrier,Pequeño
158,159,1,Norfolk Terrier,Pequeño
168,169,1,Pekingese,Pequeño
33,34,1,Bloodhound,Grande
